# This project uses an **LSTM-based Recurrent Neural Network (RNN)** to predict the next word in a sequence using a Nepal-focused text corpus. The model learns word patterns from previous context and generates the most likely next word.


In [4]:
import numpy as np 
import pandas as pd 
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Input, Dropout
from tensorflow.keras.utils import to_categorical

In [5]:
with open('../data/About Nepal.txt', 'r') as f: 
    text = f.read()

In [6]:
tokenzier = Tokenizer(lower=True)
tokenzier.fit_on_texts([text])

In [7]:
tokenzier.word_index

{'and': 1,
 'the': 2,
 'of': 3,
 'is': 4,
 'nepal': 5,
 'in': 6,
 'communities': 7,
 'a': 8,
 'are': 9,
 'can': 10,
 'to': 11,
 'important': 12,
 'many': 13,
 'cultural': 14,
 'for': 15,
 'with': 16,
 'an': 17,
 'people': 18,
 'language': 19,
 'traditional': 20,
 'community': 21,
 'festivals': 22,
 'have': 23,
 'has': 24,
 'traditions': 25,
 'food': 26,
 'indigenous': 27,
 'it': 28,
 'as': 29,
 'its': 30,
 'different': 31,
 'one': 32,
 'region': 33,
 'associated': 34,
 'technology': 35,
 'other': 36,
 'country': 37,
 'also': 38,
 'be': 39,
 'model': 40,
 'word': 41,
 'known': 42,
 'from': 43,
 'their': 44,
 'languages': 45,
 'music': 46,
 'development': 47,
 'text': 48,
 'local': 49,
 'mountains': 50,
 'contains': 51,
 'agriculture': 52,
 'tourism': 53,
 'areas': 54,
 'culture': 55,
 'natural': 56,
 'kathmandu': 57,
 'architecture': 58,
 'practices': 59,
 'knowledge': 60,
 'diverse': 61,
 'activities': 62,
 'own': 63,
 'on': 64,
 'education': 65,
 'data': 66,
 'most': 67,
 'parts': 68,

In [8]:
input_sequence = []

for sentence in text.split('.'): 
    tokenized_sentence = tokenzier.texts_to_sequences([sentence])[0]

    for i in range(1, len(tokenized_sentence)): 
        n_gram = tokenized_sentence[:i+1]
        input_sequence.append(n_gram)


In [9]:
input_sequence

[[5, 4],
 [5, 4, 8],
 [5, 4, 8, 141],
 [5, 4, 8, 141, 37],
 [5, 4, 8, 141, 37, 123],
 [5, 4, 8, 141, 37, 123, 6],
 [5, 4, 8, 141, 37, 123, 6, 256],
 [5, 4, 8, 141, 37, 123, 6, 256, 539],
 [28, 4],
 [28, 4, 8],
 [28, 4, 8, 540],
 [28, 4, 8, 540, 37],
 [28, 4, 8, 540, 37, 541],
 [28, 4, 8, 540, 37, 541, 74],
 [28, 4, 8, 540, 37, 541, 74, 358],
 [28, 4, 8, 540, 37, 541, 74, 358, 11],
 [28, 4, 8, 540, 37, 541, 74, 358, 11, 2],
 [28, 4, 8, 540, 37, 541, 74, 358, 11, 2, 257],
 [28, 4, 8, 540, 37, 541, 74, 358, 11, 2, 257, 1],
 [28, 4, 8, 540, 37, 541, 74, 358, 11, 2, 257, 1, 359],
 [28, 4, 8, 540, 37, 541, 74, 358, 11, 2, 257, 1, 359, 11],
 [28, 4, 8, 540, 37, 541, 74, 358, 11, 2, 257, 1, 359, 11, 2],
 [28, 4, 8, 540, 37, 541, 74, 358, 11, 2, 257, 1, 359, 11, 2, 256],
 [28, 4, 8, 540, 37, 541, 74, 358, 11, 2, 257, 1, 359, 11, 2, 256, 542],
 [28, 4, 8, 540, 37, 541, 74, 358, 11, 2, 257, 1, 359, 11, 2, 256, 542, 1],
 [28,
  4,
  8,
  540,
  37,
  541,
  74,
  358,
  11,
  2,
  257,
  1,
  359,

In [10]:
len_of_sequence = [len(x) for x in input_sequence]

In [11]:
max_len = max(len_of_sequence)
max_len

27

In [12]:
# Padding 
padded_input_sequences = pad_sequences(input_sequence, maxlen=max_len, padding='pre')

In [13]:
padded_input_sequences

array([[  0,   0,   0, ...,   0,   5,   4],
       [  0,   0,   0, ...,   5,   4,   8],
       [  0,   0,   0, ...,   4,   8, 141],
       ...,
       [  0,   0,   0, ...,   2,  14, 388],
       [  0,   0,   0, ...,  14, 388,   3],
       [  0,   0,   0, ..., 388,   3,   5]], shape=(4524, 27), dtype=int32)

In [14]:
X = padded_input_sequences[:, :-1]
X

array([[  0,   0,   0, ...,   0,   0,   5],
       [  0,   0,   0, ...,   0,   5,   4],
       [  0,   0,   0, ...,   5,   4,   8],
       ...,
       [  0,   0,   0, ...,  11,   2,  14],
       [  0,   0,   0, ...,   2,  14, 388],
       [  0,   0,   0, ...,  14, 388,   3]], shape=(4524, 26), dtype=int32)

In [15]:
y = padded_input_sequences[:,-1]
y

array([  4,   8, 141, ..., 388,   3,   5], shape=(4524,), dtype=int32)

In [16]:
from tensorflow.keras.utils import to_categorical
y = to_categorical(y, num_classes=len(tokenzier.word_index)+1)

In [17]:
y.shape

(4524, 1078)

In [18]:
# kerasTuner Implemention 
import keras_tuner as kt
from tensorflow.keras.optimizers import Adam

In [19]:
def build_model(hp): 
    model = Sequential()
    model.add(Input(shape=((X.shape[1], ))))

    #tune embedding dimensiona
    embedding_dim = hp.Choice(
        'embedding_dim', 
        values=[64,100,128,200]
    )
    model.add(Embedding(input_dim=len(tokenzier.word_index)+1, output_dim=embedding_dim))
    #Tune LSTM units 
    lstm_units = hp.Choice(
        'lstm_units', 
        values=[64, 100, 150, 200, 256]
    )
    model.add(LSTM(lstm_units))

    #Tune dropout 
    dropout = hp.Float(
        'dropout',
        min_value= 0.0, 
        max_value=0.5,
        step=0.1
    )
    model.add(Dropout(dropout))
    model.add(Dense(len(tokenzier.word_index) + 1, activation='softmax'))
    #Tune learning rate 
    learning_rate = hp.Choice(
        'learning_rate',
        values=[0.001, 0.005, 0.0001]
    )
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss= "categorical_crossentropy", metrics=['accuracy'])

    return model

In [20]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy', 
    max_trials=5, 
    directory='tunner',
    project_name='nepal_next_word_lsmt1', 
    overwrite=True
)

In [21]:
tuner.search(
    X,
    y,
    epochs=50,
    validation_split=0.2,
    batch_size=32
)

Trial 5 Complete [00h 03m 42s]
val_accuracy: 0.1016574576497078

Best val_accuracy So Far: 0.22983425855636597
Total elapsed time: 00h 16m 23s


In [22]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0].values
best_hp

{'embedding_dim': 128,
 'lstm_units': 150,
 'dropout': 0.30000000000000004,
 'learning_rate': 0.005}

In [23]:
print("Best Embedding Dimension:",
      best_hp.get("embedding_dim"))

print("Best LSTM Units:",
      best_hp.get("lstm_units"))

print("Best Dropout:",
      best_hp.get("dropout"))

print("Best Learning Rate:",
      best_hp.get("learning_rate"))

Best Embedding Dimension: 128
Best LSTM Units: 150
Best Dropout: 0.30000000000000004
Best Learning Rate: 0.005


In [24]:
best_model = tuner.get_best_models(num_models=1)[0]

C:\Users\user\anaconda3\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(store)


In [25]:
best_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 26, 128)             │         137,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ (None, 150)                 │         167,400 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 150)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 1078)                │         162,778 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 468,162 (1.79 MB)

 Trainable params: 468,162 (1.79 MB)

 Non-trainable params: 0 (0.00 B)

In [30]:
def is_related(text):
    words = text.lower().split()
    known = [w for w in words if w in tokenzier.word_index and w != "<OOV>"]
    return len(known) > len(words) / 2

In [36]:
def predict_text(text, model, tokenizer):
    if not is_related(text):
        return f"[Not related to corpus] '{text}' has no words from the Nepal corpus."


    for i in range(20):

        # Tokenize
        token_text = tokenizer.texts_to_sequences([text])[0]

        # Padding
        padded_text = pad_sequences(
            [token_text],
            maxlen=len(tokenizer.word_index) + 1,
            padding='pre'
        )

        # Predict
        prediction = model.predict(padded_text, verbose=0)

        # Get word position
        pos = np.argmax(prediction)

        # Find the word
        for word, index in tokenizer.word_index.items():
            if index == pos:
                text = text + " " + word
                break

    return text

In [32]:


result = predict_text("pokhara", best_model, tokenzier)

print(result)

pokhara is one of the most popular sports in nepal and other countries visit lumbini for religious cultural historical and educational


In [33]:


result = predict_text("Nepal", best_model, tokenzier)

print(result)

Nepal has a unique national flag and bhaktapur an a large and diverse corpus can help a model learn vocabulary sentence


In [34]:

result = predict_text("culture", best_model, tokenzier)

print(result)

culture provide a balance between predictable and diverse generation can help a balance between predictable and diverse generation and is a


In [35]:

result = predict_text("fuck", best_model, tokenzier)

print(result)

NameError: name 'seed' is not defined

In [ ]:
import joblib

In [ ]:

# Save model
best_model.save("next_word_lstm.keras")

# Save tokenizer
joblib.dump(tokenzier, "tokenzier.pkl")

print("Saved successfully!") 

In [ ]:
result = predict_text("Indigenous", best_model, tokenzier)
result